# 04 — All figures as PowerPoint and PDF

Run after 01–03 (they save every figure to `results/figures/` as PDF, PNG and SVG;
02 and 03 read cached simulations, so re-running them only redraws the figures).

Output in `results/`:

| File | Content |
|---|---|
| `Figures_main.pptx`, `Figures_supplementary.pptx` | one slide per figure, at its real size; legend in the speaker notes |
| `Figures_main.pdf`, `Figures_supplementary.pdf` | the same, one figure per page (vector) |
| `figure_legends.txt` | all legends, ready to paste into the manuscript |

**Editing in PowerPoint.** Each figure is placed as an SVG (vector) with a PNG fallback.
Right-click → *Convert to Shape* (PowerPoint 365 / 2019 or later) makes every line and every
text box editable. Fonts are Arial at the sizes used in the paper.

**Figures drawn by hand.** Fig. 2 (flowchart) and Fig. S7 (model DAG) are not made by code.
Save them as `figures_manual/Fig2_one_cycle.svg` (or `.png`) and
`figures_manual/FigS7_model_DAG.svg` (or `.png`); otherwise a placeholder slide is inserted.

Legends and take-home messages are edited in `campyevo/figure_deck.py`.

In [ ]:
import sys, os, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import campyevo
from campyevo import n, QUICK, DATA, RESULTS, FIGURES, TABLES, SIMS, POSTERIOR
from campyevo.plotting import setup_style, savefig
setup_style()
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
warnings.simplefilter("ignore", RuntimeWarning)
print("project root:", ROOT, "| QUICK mode:", QUICK)

In [ ]:
# Packages for this notebook only (python-pptx is imported as "pptx").
# Missing ones are installed with conda (conda-forge) into the environment of this kernel.
import sys, os, importlib, importlib.util
NEEDED = {"pptx": "python-pptx", "pypdf": "pypdf", "PIL": "pillow"}
missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    if os.path.isdir(os.path.join(sys.prefix, "conda-meta")):      # the kernel runs in a conda environment
        print("installing with conda:", " ".join(missing), "->", sys.prefix)
        %conda install -y -q -c conda-forge {" ".join(missing)}
        importlib.invalidate_caches()
    else:
        raise RuntimeError("this kernel is not a conda environment; in a terminal run\n"
                           f"  conda activate campyevo\n  conda install -c conda-forge {' '.join(missing)}\n"
                           "then restart the kernel")
print("ok:", {mod: importlib.util.find_spec(mod) is not None for mod in NEEDED})

In [ ]:
from campyevo import figure_deck as FD
MANUAL = ROOT / "figures_manual"
MANUAL.mkdir(exist_ok=True)
LABEL_SLIDES = True     # False: no figure number / message above the figure on each slide

rep_main = FD.build_pptx(FD.MAIN, FIGURES, RESULTS / "Figures_main.pptx", manual_dir=MANUAL,
                         label_slides=LABEL_SLIDES)
rep_sup = FD.build_pptx(FD.SUPPLEMENTARY, FIGURES, RESULTS / "Figures_supplementary.pptx",
                        manual_dir=MANUAL, label_slides=LABEL_SLIDES)
pd.DataFrame(rep_main + rep_sup)

In [ ]:
miss = FD.build_pdf(FD.MAIN, FIGURES, RESULTS / "Figures_main.pdf", manual_dir=MANUAL)
miss += FD.build_pdf(FD.SUPPLEMENTARY, FIGURES, RESULTS / "Figures_supplementary.pdf", manual_dir=MANUAL)
FD.write_legends(FD.ALL, RESULTS / "figure_legends.txt")
print("not in the PDFs (no PDF file):", miss or "none")
print((RESULTS / "figure_legends.txt").read_text()[:1500])